# Behavioural Features for Payments Fraud (PaySim)
**Author:** Jesly Prosper

## Why a second dataset
My first fraud model used the Kaggle credit card dataset, whose features are anonymised PCA components. That makes it impossible to build the **behavioural features** real fraud systems rely on, such as how often an account is paid or whether a payee is brand new.

[PaySim](https://www.kaggle.com/datasets/ealaxi/paysim1) is a simulated mobile-money dataset of 6.3 million transactions with real account IDs, transaction types and balances. Here I measure **how much behavioural features improve fraud detection**, compared with using raw transaction fields alone.

**Honest caveat:** PaySim is simulated from real logs, not real customer data, so some patterns (especially in balances) are cleaner than they would be in reality. I test those separately so they don't inflate the main result.

## Plan
1. Explore the data and the existing rule (`isFlaggedFraud`)
2. Build three feature sets: **raw**, **raw + behavioural**, and **raw + behavioural + balance checks**
3. Compute every behavioural feature from **past transactions only**, so nothing leaks from the future
4. Train on earlier hours and test on later ones
5. Compare the feature sets with PR-AUC and fraud caught within a fixed alert budget

In [ ]:
import glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score
RANDOM_STATE = 42
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Load the data

In [ ]:
files = [f for f in glob.glob('/kaggle/input/**/*.csv', recursive=True) + glob.glob('*.csv') + glob.glob('data/*.csv')
         if 'isFraud' in open(f).readline()]
if not files:
    raise FileNotFoundError("PaySim CSV not found. Click 'Add Input', search 'paysim1' (by ealaxi) and add it.")
dtypes = {'step': 'int16', 'type': 'category', 'amount': 'float64', 'nameOrig': 'string', 'oldbalanceOrg': 'float64',
          'newbalanceOrig': 'float64', 'nameDest': 'string', 'oldbalanceDest': 'float64', 'newbalanceDest': 'float64',
          'isFraud': 'int8', 'isFlaggedFraud': 'int8'}
df = pd.read_csv(files[0], dtype=dtypes)
print(f'Loaded {len(df):,} transactions over {df.step.max()} hours from {files[0]}')
df.head()

## 2. Explore

In [ ]:
print(f"Fraud: {df.isFraud.sum():,} of {len(df):,} ({df.isFraud.mean():.3%})")
summary = df.groupby('type', observed=True).agg(transactions=('isFraud', 'size'), frauds=('isFraud', 'sum'))
summary['fraud_rate'] = summary.frauds / summary.transactions
summary

In [ ]:
# The dataset's built-in rule flags transfers over 200,000. How well does it do?
rule = df.isFlaggedFraud == 1
print(f"Existing rule flags {rule.sum():,} transactions and catches {df.loc[rule, 'isFraud'].sum():,} "
      f"of {df.isFraud.sum():,} frauds ({df.loc[rule, 'isFraud'].sum() / df.isFraud.sum():.2%})")

Fraud only happens in **TRANSFER** and **CASH_OUT** transactions, and the built-in rule catches almost none of it. I'll model those two types, but compute account history from **all** transactions, because a payee's earlier activity of any type is useful context.

## 3. Behavioural features (past transactions only)
For each transaction I look back at the **same payee** (`nameDest`) and the **same payer** (`nameOrig`), counting only transactions in **earlier hours**, so the model never sees the future.

In [ ]:
def history_features(data, key_col, prefix):
    """Per-account history using only earlier hours: prior count, count in last 24h, hours since last, prior mean amount."""
    d = data[[key_col, 'step', 'amount']].copy()
    d['_row'] = np.arange(len(d))
    d = d.sort_values([key_col, 'step'], kind='stable')
    gid = pd.factorize(d[key_col])[0].astype(np.int64)
    step = d['step'].to_numpy(np.int64)
    key = gid * 10_000 + step                       # sorted, and groups never overlap (steps < 10,000)
    first_same = np.searchsorted(key, key, 'left')  # first row for this account in this hour
    grp_start = np.searchsorted(key, gid * 10_000, 'left')
    win_start = np.searchsorted(key, key - 24, 'left')
    prior = first_same - grp_start
    csum = np.r_[0.0, np.cumsum(d['amount'].to_numpy())]
    prior_sum = csum[first_same] - csum[grp_start]
    last = first_same - 1
    has_prev = last >= grp_start
    since = np.where(has_prev, step - step[np.clip(last, 0, None)], -1)
    out = pd.DataFrame({
        f'{prefix}_prior_count': prior,
        f'{prefix}_count_24h': first_same - win_start,
        f'{prefix}_hours_since_last': since,
        f'{prefix}_is_new': (prior == 0).astype(np.int8),
        f'{prefix}_prior_mean_amount': np.where(prior > 0, prior_sum / np.maximum(prior, 1), np.nan),
    }, index=d['_row'].to_numpy()).sort_index()
    return out.set_axis(data.index)

df = df.sort_values('step', kind='stable').reset_index(drop=True)
df = pd.concat([df, history_features(df, 'nameDest', 'dest'), history_features(df, 'nameOrig', 'orig')], axis=1)
df['hour_of_day'] = df.step % 24
df['amount_vs_dest_mean'] = df.amount / df.dest_prior_mean_amount
df['drains_origin'] = ((df.oldbalanceOrg > 0) & (df.amount >= df.oldbalanceOrg)).astype(np.int8)

# Balance consistency checks (strong in this simulator, so tested separately)
df['error_orig'] = df.newbalanceOrig + df.amount - df.oldbalanceOrg
df['error_dest'] = df.oldbalanceDest + df.amount - df.newbalanceDest

data = df[df.type.isin(['TRANSFER', 'CASH_OUT'])].copy()
data['is_transfer'] = (data.type == 'TRANSFER').astype(np.int8)
data['log_amount'] = np.log1p(data.amount)
print(f"Modelling {len(data):,} TRANSFER and CASH_OUT transactions, {data.isFraud.sum():,} frauds ({data.isFraud.mean():.2%})")
data[['dest_prior_count', 'dest_count_24h', 'dest_hours_since_last', 'dest_is_new', 'amount_vs_dest_mean', 'drains_origin']].describe().T

In [ ]:
# Does behaviour differ for fraud? Compare averages by class.
compare = ['dest_is_new', 'dest_prior_count', 'dest_count_24h', 'drains_origin', 'hour_of_day', 'orig_prior_count']
data.groupby('isFraud')[compare].mean().T.rename(columns={0: 'Genuine', 1: 'Fraud'})

## 4. Time-based split
Train on the earliest 60% of transactions, validate on the next 20%, and test on the latest 20%.

In [ ]:
n = len(data); i1, i2 = int(n * 0.6), int(n * 0.8)
train, val, test = data.iloc[:i1], data.iloc[i1:i2], data.iloc[i2:]
for name, part in [('Train', train), ('Validation', val), ('Test', test)]:
    print(f"{name:<11} hours {part.step.min():>3} to {part.step.max():>3}  {len(part):>9,} rows  fraud {part.isFraud.sum():>5,} ({part.isFraud.mean():.2%})")

## 5. Compare feature sets

In [ ]:
RAW = ['log_amount', 'is_transfer', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
BEHAVIOURAL = ['hour_of_day', 'dest_prior_count', 'dest_count_24h', 'dest_hours_since_last', 'dest_is_new',
               'dest_prior_mean_amount', 'amount_vs_dest_mean', 'orig_prior_count', 'drains_origin']
BALANCE = ['error_orig', 'error_dest']
feature_sets = {'Raw fields': RAW, 'Raw + behavioural': RAW + BEHAVIOURAL, 'Raw + behavioural + balance checks': RAW + BEHAVIOURAL + BALANCE}
ALERT_BUDGET = 0.01  # analysts can review 1% of transactions

def fraud_caught_at_budget(y, p, budget=ALERT_BUDGET):
    k = max(1, int(len(p) * budget))
    top = np.argsort(-p)[:k]
    return y[top].sum() / y.sum()

results, models, scores = {}, {}, {}
for name, cols in feature_sets.items():
    m = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=300, max_leaf_nodes=63, l2_regularization=1.0,
                                       early_stopping=True, validation_fraction=0.1, random_state=RANDOM_STATE)
    m.fit(train[cols], train.isFraud)
    p = m.predict_proba(test[cols])[:, 1]
    y = test.isFraud.to_numpy()
    results[name] = {'PR-AUC': average_precision_score(y, p), 'ROC-AUC': roc_auc_score(y, p),
                     f'Fraud caught at {ALERT_BUDGET:.0%} alert budget': fraud_caught_at_budget(y, p)}
    models[name], scores[name] = m, p
res = pd.DataFrame(results).T
res

In [ ]:
base, beh = res.loc['Raw fields'], res.loc['Raw + behavioural']
print(f"Behavioural features raised PR-AUC from {base['PR-AUC']:.3f} to {beh['PR-AUC']:.3f} "
      f"and fraud caught at a {ALERT_BUDGET:.0%} alert budget from {base.iloc[2]:.1%} to {beh.iloc[2]:.1%}.")
ax = res['PR-AUC'].plot.barh(color=['#86868b', '#1F3864', '#C0392B'], figsize=(7, 3))
ax.set_xlabel('PR-AUC on the test period (higher is better)'); ax.set_xlim(0, 1); plt.tight_layout(); plt.show()

## 6. Which behavioural features matter?
Permutation importance on the raw + behavioural model: how much PR-AUC drops when each feature is shuffled.

In [ ]:
cols = feature_sets['Raw + behavioural']
sample = pd.concat([test[test.isFraud == 1], test[test.isFraud == 0].sample(min(50_000, (test.isFraud == 0).sum()), random_state=RANDOM_STATE)])
imp = permutation_importance(models['Raw + behavioural'], sample[cols], sample.isFraud, scoring='average_precision',
                             n_repeats=3, random_state=RANDOM_STATE)
top = pd.Series(imp.importances_mean, index=cols).sort_values().tail(10)
colors = ['#1F3864' if c in BEHAVIOURAL else '#86868b' for c in top.index]
top.plot.barh(color=colors, figsize=(7, 4)); plt.title('Top features (blue = behavioural)'); plt.xlabel('Drop in PR-AUC'); plt.tight_layout(); plt.show()

## 7. Conclusions
- The results table and the sentence printed above give the measured gain from behavioural features on a **later, unseen time period**.
- Behavioural features use only **past** transactions, the same constraint a live scoring system has.
- The balance-check features are very strong here, but they reflect how the simulator generates balances, so I report the **raw + behavioural** model as the realistic result.

**Next steps:** add rolling amount features per payer, try graph features linking payers and payees (mule accounts), and serve the model through the scoring API in this repo.